# 实验六 · YUV420 → RGB（综合案例）

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐⭐ 综合（capstone）　|　**预计时长**：40–50 分钟

> **实验说明**
> 1. 本实验是本章的**综合案例**，将前几个实验的技术整合于一条完整的视频解码流程之中。采用分步实现的方式：由 **v1 串行定点版本**起，依次引入 **v2 NEON 完整流程**与 **v3 循环展开**。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 三个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方式，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验综合运用**色度上采样、类型提升、定点运算、饱和窄化与无分支处理**，建议先完成前述全部实验。

## 🎯 学习目标

完成本实验后，学生应能够：

- 理解 **YUV420(4:2:0)** 的数据格式与色度子采样
- 掌握 **ITU-R BT.601** 转换公式及其**定点化**形式
- 掌握**色度上采样**（`vzip` 交织复制）、**类型提升**（`vmovl`）、**定点乘加**（`vmlaq`）与**饱和窄化**（`vqmovun`，无分支钳位）
- 通过 **Serial Float 与 Serial Int 的对比**，理解一个关键结论：**定点化在标量层面不但不省时间、甚至可能略慢，其价值在于使向量化成为可能**
- 综合运用本章全部技术，独立分析一条完整图像处理流程的性能

## 🗺️ 学习路径

1. **准备阶段**：理解 YUV420 数据格式与 BT.601 转换公式（浮点 → 定点）
2. **v1 · 浮点参考 + 串行定点**：实现 `yuv420_to_rgb_float`（浮点，作为精度参考）与 `yuv420_to_rgb_int`（串行定点，作为加速比基准）
   → 对比二者耗时，考察“定点化在标量层面是否省时间”
3. **v2 · NEON 完整流程**：新增 `yuv420_to_rgb_neon`，整合上采样、类型提升、定点乘加与饱和窄化
   → 掌握综合流程的向量化实现
4. **v3 · 循环展开**：新增 `yuv420_to_rgb_neon_unroll`
   → 考察循环展开的进一步收益
5. **可视化与分析**：以 v3 的输出结果绘制加速比柱状图，得出“定点化为向量化服务”的结论

## 1. 背景与动机

视频编解码与相机采集普遍采用 **YUV** 而非 RGB 表示颜色。YUV 是一种颜色编码方法，其中 **Y** 表示明亮度（Luma/Luminance），**U（Cb）** 与 **V（Cr）** **都是色度（Chroma）分量**——前者是**蓝色差** $B-Y$、后者是**红色差** $R-Y$，因此该编码也称为 **YCbCr**。

采用 YUV 的根本原因在于**人眼对亮度敏感、对色度不敏感**这一视觉特性。据此可对色度分量降采样，在几乎不损失主观画质的前提下节省存储与带宽。

### 1.1 存储格式：紧缩与平面

YUV 数据按存储格式不同可分为两类：

- **紧缩格式（packed formats）**：将同一像素的 Y、U、V 值交织存储为 Macro Pixels 数组，三个分量在内存中彼此相邻。
- **平面格式（planar formats）**：将 Y、U、V 三个分量分别存放于不同的矩阵（平面）中，各平面在内存中连续。

本实验采用**平面格式**：Y 平面为全分辨率 $W\times H$，U、V 平面各为 $\frac{W}{2}\times\frac{H}{2}$。平面格式便于将同一分量的数据连续加载至向量寄存器，是向量化处理的常见前提。

### 1.2 色度子采样与 J:a:b 表示法

色度子采样（chroma subsampling）依据人眼对色度不敏感的特性，减少色度采样点以节省内存与带宽。其规模常以 **J:a:b** 表示法描述：

- **J**：参考块的宽度（列数），通常取 4；
- **a**：第一行中色度采样点的个数；
- **b**：第二行中色度采样点的个数（`0` 表示与第一行共用，即第二行不再单独采样）。

本实验采用的 **YUV420（4:2:0）** 表示：在 $4\times2$ 的参考块中，第一行采样 2 个色度点，第二行与第一行共用（`b=0`）。其效果等价于在水平与垂直方向上各将色度降采样一半——即每 $2\times2$ 个 Y 像素共享一组 $(U, V)$，色度数据量仅为原先的 **1/4**。

![](images/03.06_jab_chroma_subsampling.png)

### 1.3 YUV420 数据格式与对齐

YUV420 依 U、V 的排布方式又细分为多种具体格式：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">大类</th>
      <th style="text-align: left;">具体格式</th>
      <th style="text-align: left;">U、V 的排布</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>YUV420P</strong>（平面）</td>
      <td style="text-align: left;">YU12（I420）</td>
      <td style="text-align: left;">U 平面在前、V 平面在后</td>
    </tr>
    <tr>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">YV12</td>
      <td style="text-align: left;">V 平面在前、U 平面在后</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>YUV420SP</strong>（半平面）</td>
      <td style="text-align: left;">NV12</td>
      <td style="text-align: left;">U、V 交织为一个平面（UVUV…）</td>
    </tr>
    <tr>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">NV21</td>
      <td style="text-align: left;">V、U 交织为一个平面（VUVU…）</td>
    </tr>
  </tbody>
</table>

> **数据对齐问题**：向量化要求 **Y 与 U/V 在通道上一一对应**。由于 4:2:0 中每组 $(U,V)$ 被 $2\times2$ 个 Y 共享，色度分量的数量仅为亮度的 1/4，直接加载无法与 8 个 Y 对齐。因此在进入转换公式前，必须先将 U、V **上采样**补齐到与每个像素一一对应——这正是本实验 NEON 流程的第一个关键步骤（见第 3 节）。

![](images/03.06_yuv420.png)

**本实验的数据布局（planar，I420 风格）**：输入为 Y 平面（`pixels` 字节）+ U 平面（`pixels/4`）+ V 平面（`pixels/4`），共 `pixels × 3/2` 字节；输出为 R、G、B 三个平面，共 `pixels × 3` 字节。


## 2. 算法与公式

本实验采用 **ITU-R BT.601** 标准（SDTV，标准清晰度电视）的反变换，将 YUV 还原为 RGB。

### 2.1 浮点公式

$$R = 1.164\cdot(Y-16) + 1.596\cdot(V-128)$$
$$G = 1.164\cdot(Y-16) - 0.391\cdot(U-128) - 0.813\cdot(V-128)$$
$$B = 1.164\cdot(Y-16) + 2.018\cdot(U-128)$$

其中 $Y-16$ 用于将有限范围（video range，$Y\in[16,235]$）的亮度归位，$U-128$、$V-128$ 将色度归位到以 0 为中心。

### 2.2 定点化：浮点 → 整数（Q8）

浮点运算不利于向量化。将系数统一乘以 $256$（即 Q8 定点，等价于左移 8 位）后取整，即可用整数乘加代替浮点。下表沿用业界通用的一组 BT.601 定点常数：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">浮点系数</th>
      <th style="text-align: left;">×256 取整（Q8）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">1.164</td>
      <td style="text-align: left;">≈ 298</td>
    </tr>
    <tr>
      <td style="text-align: left;">1.596</td>
      <td style="text-align: left;">≈ 409</td>
    </tr>
    <tr>
      <td style="text-align: left;">0.391</td>
      <td style="text-align: left;">≈ 100</td>
    </tr>
    <tr>
      <td style="text-align: left;">0.813</td>
      <td style="text-align: left;">≈ 208</td>
    </tr>
    <tr>
      <td style="text-align: left;">2.018</td>
      <td style="text-align: left;">≈ 516</td>
    </tr>
  </tbody>
</table>

先对分量归位：
$$C = Y-16,\quad D = U-128,\quad E = V-128$$
得到定点公式（系数已乘以 256）：
$$r = (298C + 409E + 128)\gg 8$$
$$g = (298C - 100D - 208E + 128)\gg 8$$
$$b = (298C + 516D + 128)\gg 8$$

其中 `+128` 相当于加上 $0.5\times256$，用于右移前的四舍五入；`>>8`（即除以 256）抵消系数放大的倍率。

### 2.3 钳位（clamp / 饱和）

定点乘加的中间结果可能超出 $[0,255]$，最终须**钳位**到合法像素范围：
$$R = \mathrm{clamp}(r,\, 0,\, 255)\quad(\text{G、B 同理})$$

> 补充：R 仅与 E（Cr）有关，B 仅与 D（Cb）有关，而 G 同时减去 D 与 E——因为绿色分量在 YUV 中没有独立的色差，只能由亮度扣除红、蓝的贡献反推得到，故 G 的公式最复杂。


## 3. 核心 NEON 指令与技巧

本实验的 NEON 流程由四个关键步骤组成，分别对应源码中的四类指令。理解每一步"为什么这样做"，比记住指令名称更为重要。

### ① 色度上采样（`vzip` 交织复制）

如第 1.3 节所述，4:2:0 中每 4 个 U/V 对应 8 个 Y，须先把 **4 个 U/V 扩展为 8 个**，才能与 8 个 Y 对齐。

- **传统做法**：移位、掩码、标量逐个拷贝，指令延迟高、并行度低。
- **`vzip` 交织复制**：将**同一个寄存器作为两个输入**送入交织指令，由硬件一步完成"双倍复制"。

```c
uint8x8_t u = vzip_u8(u_full, u_full).val[0];  // [U0,U0,U1,U1,U2,U2,U3,U3]
```

即把 `[U0,U1,U2,U3,…]` 与自身交织，取低半部分即得到每个元素重复一次的结果，恰好完成 2 倍上采样。

### ② 类型提升（`vmovl`，防溢出）

**溢出风险**：定点乘加的结果会远超 8 位甚至 16 位上限。以 R 为例，取极端输入 $Y=255$、$V=255$，即归位后 $C=Y-16=239$、$E=V-128=127$：
$$298C + 409E + 128 = 298\times239 + 409\times127 + 128 = 123293$$
该值已远超 16 位有符号数上限（32767），若不加宽寄存器位宽必然溢出。因此必须**逐级加宽**，为乘加结果留出空间：

- `vmovl_u8`：8 位 → 16 位（配合 `vsubq_s16` 完成分量归位）
- `vmovl_s16`：16 位 → 32 位（为 32 位乘加做准备）

```c
int16x8_t C = vsubq_s16(vreinterpretq_s16_u16(vmovl_u8(y_u8)), v16);  // u8 -> s16
int32x4_t c_lo = vmovl_s16(vget_low_s16(C));                          // s16 -> s32
```

### ③ 定点乘加（`vmlaq_s32`）

在 32 位空间内以整数乘加累加 $298C + 409E + 128$，一条指令即可完成"乘 + 加"：

```c
int32x4_t r = vmlaq_s32(v128_32, c, c298);  // 128 + 298*C
r = vmlaq_s32(r, e, c409);                   // + 409*E
```

### ④ 饱和窄化与无分支处理（`vshrn` + `vqmovun`）

这是本实验最具代表性的技巧，需理解其性能动机。

**普通边界钳位的隐性瓶颈**：标量钳位通常写成两个 `if` 判断：

```c
static inline uint8_t clamp_u8(int x){
  if (x < 0)   return 0;
  if (x > 255) return 255;
  return (uint8_t)x;
}
```

在超标量、深流水线的 CPU 上，数据相关的分支是隐性瓶颈：像素是否越界高度依赖输入数据，**分支预测器命中率下降**；一旦预测失败，将触发**流水线冲刷（Pipeline Flush）**，每次浪费约 **10–20 个周期**。处理 1080P（约 200 万像素）时，惩罚会线性累积。

> 🔬 **不过要留个心眼：编译器可能已经替你消掉了一部分分支。** 用 `gcc -O3 -S` 观察 `yuv420_to_rgb_int` 的内层循环会发现，GCC把上面这段 `clamp_u8` 编成了：
>
> ```asm
> tbnz  w1, #31, .L31       ← 只剩这一个真分支：测符号位，x<0 时跳过（结果寄存器预置为 0）
> cmp   w23, 256
> csel  w7, w26, w8, lt     ← 上界钳位已经变成无分支的条件选择
> ```
>
> 也就是说实际编译结果是"**一次分支 + 一次条件选择**"，而非"两次分支"，请以反汇编为准。这不影响 `vqmovun` 的价值（它把最后这个分支也消掉了，而且一次处理 8 个像素）。

**无分支编程**：用两条 NEON 指令替代 `if`，将钳位固化进指令本身，从而彻底消除分支：

- **`vshrn`**：右移与窄化一步完成（shift + narrow）。
- **`vqmovun`**：带饱和的窄化，把钳位直接固化进指令（`<0→0`、`>255→255`）。其命名可拆解为 **q**（饱和 saturate）· **mov**（移动）· **u**（转无符号 to unsigned）· **n**（窄化 narrow）。

```c
int16x8_t r_16 = vcombine_s16(vshrn_n_s32(r_lo, 8), vshrn_n_s32(r_hi, 8));  // >>8 并窄化
vst1_u8(r_out + i, vqmovun_s16(r_16));                                      // 饱和到 u8
```

如此，钳位过程**零 CMP、零 Branch**——不再有分支预测失败，流水线也不会因此产生气泡。

> 通读 v2 的源码时，请将这四个步骤逐一对应到代码中，理解它们如何串联成一条完整的处理流程：**上采样 → 类型提升 → 定点乘加 → 饱和窄化输出**。


## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows

In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 创建源代码目录
!mkdir -p src_yuv2rgb

## 5. v1 · 浮点参考与串行定点实现

与前面实验略有不同：本案例的两个串行版本并非“是否关闭向量化”的关系，而是**浮点与定点**的关系：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>yuv420_to_rgb_float</code></td>
      <td style="text-align: left;">采用<strong>浮点</strong> BT.601 公式，作为<strong>精度参考</strong>（标记为 REF），后续版本的正确性以其为准</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>yuv420_to_rgb_int</code></td>
      <td style="text-align: left;">采用<strong>定点</strong>（Q8 整数）公式，是<strong>串行基准</strong>（1.00×），后续加速比以其为参照</td>
    </tr>
  </tbody>
</table>

### 💡 关注点：定点化在标量层面是否省时间
请在运行后**对比 `Serial Float` 与 `Serial Int` 的耗时**。多数情况下二者相差不大——现代处理器配有硬件浮点单元（FPU），标量层面的浮点乘加与整数乘加吞吐相近。这说明**定点化本身并不是为了在标量上提速**；其真正价值将在 v2 中显现。

### 数据布局
- 输入 `yuv` 为 planar 格式（Y + U + V），共 `pixels × 3/2` 字节；输出 `rgb` 为 R/G/B 三平面
- 参考结果 `rgb_ref` 由浮点版本生成；`check_diff` 容差取 2（浮点与定点近似带来的差异）

In [ ]:
%%writefile src_yuv2rgb/yuv2rgb_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Clamp an integer to the u8 range [0, 255]
static inline uint8_t clamp_u8(int x) {
  if (x < 0) return 0;
  if (x > 255) return 255;
  return (uint8_t)x;
}

// Helper: Verify results against reference (allow +/-2 for float/fixed diff)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n * 3; i++) {
    int diff = abs((int)ref[i] - (int)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff <= 2)
    return "PASS";
  else
    return "FAIL";
}

// 1. Floating-point BT.601 (accurate reference)
void yuv420_to_rgb_float(const uint8_t* yuv, uint8_t* rgb, int width,
                         int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    for (int i = 0; i < width; i++) {
      int y_idx = j * width + i;
      int uv_idx = (j / 2) * (width / 2) + (i / 2);

      float y_val = Y[y_idx];
      float u_val = U[uv_idx];
      float v_val = V[uv_idx];

      // Standard BT.601 conversion
      float c = y_val - 16.0f;
      float d = u_val - 128.0f;
      float e = v_val - 128.0f;

      int r = (int)(1.164f * c + 1.596f * e);
      int g = (int)(1.164f * c - 0.391f * d - 0.813f * e);
      int b = (int)(1.164f * c + 2.018f * d);

      long out_idx = j * width + i;
      // Output: R plane, then G plane, then B plane
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// 2. Serial fixed-point (Q8 integer) - Baseline for speedup
void yuv420_to_rgb_int(const uint8_t* yuv, uint8_t* rgb, int width,
                       int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    int y_row = j * width;
    int uv_row = (j / 2) * (width / 2);

    for (int i = 0; i < width; i++) {
      int y_val = Y[y_row + i];
      int u_val = U[uv_row + (i / 2)];
      int v_val = V[uv_row + (i / 2)];

      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;

      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;

      long out_idx = y_row + i;
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*yuv_fn)(const uint8_t*, uint8_t*, int, int);

static double bench(yuv_fn fn, const uint8_t* yuv, uint8_t* rgb, int w, int h) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(yuv, rgb, w, h);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(
      " YUV420 to RGB v1: Float Reference and Serial Fixed-Point (BT.601)\n");
  printf(" Image: %d x %d   Loops: %d\n", width, height, NTIMES);
  printf("===========================================================\n");

  long pixels = (long)width * height;
  // YUV420 planar = Y(pixels) + U(pixels/4) + V(pixels/4) = pixels*3/2
  size_t bytes_yuv = ((size_t)pixels * 3 / 2 + 15) & ~(size_t)15;
  size_t bytes_rgb = ((size_t)pixels * 3 + 15) & ~(size_t)15;

  uint8_t* yuv = (uint8_t*)aligned_alloc(16, bytes_yuv);
  uint8_t* rgb_ref = (uint8_t*)aligned_alloc(16, bytes_rgb);  // float reference
  uint8_t* rgb_test = (uint8_t*)aligned_alloc(16, bytes_rgb);  // working buffer
  if (!yuv || !rgb_ref || !rgb_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialize planar YUV data
  memset(yuv, 0, pixels * 3 / 2);
  for (long i = 0; i < pixels * 3 / 2; i++) yuv[i] = (i % 255);

  // Golden reference = floating-point conversion
  yuv420_to_rgb_float(yuv, rgb_ref, width, height);

  double t_fl = bench(yuv420_to_rgb_float, yuv, rgb_ref, width, height);
  double t_in = bench(yuv420_to_rgb_int, yuv, rgb_test, width, height);
  const char* s_in = check_diff(rgb_ref, rgb_test, pixels);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial Float    | %9.3f |    -    |  REF  |\n", t_fl);
  printf("| Serial Int      | %9.3f |  1.00 x |  %-4s |\n", t_in, s_in);
  printf("-------------------------------------------------\n");

  free(yuv);
  free(rgb_ref);
  free(rgb_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_yuv2rgb/yuv2rgb_v1.c", "src_yuv2rgb/yuv2rgb_v1")
out_v1 = run_bin(BIN, 1920, 1080)

## 6. v2 · NEON 完整流程实现

在 v1 的基础上**新增 `yuv420_to_rgb_neon` 函数**，将第 3 节的四个步骤整合为一条向量化流程：**上采样 → 类型提升 → 定点乘加 → 饱和窄化**。

### 🔑 知识点
- **色度上采样（`vzip`）**：色度分辨率仅为亮度的一半，须先将 U、V 复制扩展以对齐每个 Y
- **类型提升（`vmovl`）防溢出**：`298*C` 等中间结果远超 8/16 位，须逐级加宽至 s16、s32
- **有符号运算**：色差 D、E 可正可负，故中间量采用**有符号**类型
- **饱和窄化（`vqmovun`）实现无分支钳位**：将有符号结果饱和为 u8，替代了浮点版本中的 `if` 判断，消除了**分支预测**的开销

此版本包含**三行**输出。请观察 NEON 相对 `Serial Int` 的加速比——这正是定点化所“解锁”的性能。

In [ ]:
%%writefile src_yuv2rgb/yuv2rgb_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Clamp an integer to the u8 range [0, 255]
static inline uint8_t clamp_u8(int x) {
  if (x < 0) return 0;
  if (x > 255) return 255;
  return (uint8_t)x;
}

// Helper: Verify results against reference (allow +/-2 for float/fixed diff)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n * 3; i++) {
    int diff = abs((int)ref[i] - (int)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff <= 2)
    return "PASS";
  else
    return "FAIL";
}

// 1. Floating-point BT.601 (accurate reference)
void yuv420_to_rgb_float(const uint8_t* yuv, uint8_t* rgb, int width,
                         int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    for (int i = 0; i < width; i++) {
      int y_idx = j * width + i;
      int uv_idx = (j / 2) * (width / 2) + (i / 2);

      float y_val = Y[y_idx];
      float u_val = U[uv_idx];
      float v_val = V[uv_idx];

      // Standard BT.601 conversion
      float c = y_val - 16.0f;
      float d = u_val - 128.0f;
      float e = v_val - 128.0f;

      int r = (int)(1.164f * c + 1.596f * e);
      int g = (int)(1.164f * c - 0.391f * d - 0.813f * e);
      int b = (int)(1.164f * c + 2.018f * d);

      long out_idx = j * width + i;
      // Output: R plane, then G plane, then B plane
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// 2. Serial fixed-point (Q8 integer) - Baseline for speedup
void yuv420_to_rgb_int(const uint8_t* yuv, uint8_t* rgb, int width,
                       int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    int y_row = j * width;
    int uv_row = (j / 2) * (width / 2);

    for (int i = 0; i < width; i++) {
      int y_val = Y[y_row + i];
      int u_val = U[uv_row + (i / 2)];
      int v_val = V[uv_row + (i / 2)];

      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;

      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;

      long out_idx = y_row + i;
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// 3. NEON fixed-point: vzip upsample + vmovl widen + vmla + vqmovun saturate
void yuv420_to_rgb_neon(const uint8_t* yuv, uint8_t* rgb, int width,
                        int height) {
  long pixels = width * height;
  const uint8_t* Y_ptr = yuv;
  const uint8_t* U_ptr = yuv + pixels;
  const uint8_t* V_ptr = U_ptr + (pixels / 4);

  // Constants
  const int16x8_t v16 = vdupq_n_s16(16);
  const int16x8_t v128 = vdupq_n_s16(128);
  const int32x4_t v128_32 = vdupq_n_s32(128);

  const int32x4_t c298 = vdupq_n_s32(298);
  const int32x4_t c409 = vdupq_n_s32(409);
  const int32x4_t c100_neg = vdupq_n_s32(-100);
  const int32x4_t c208_neg = vdupq_n_s32(-208);
  const int32x4_t c516 = vdupq_n_s32(516);

  int width_uv = width / 2;

  for (int j = 0; j < height; j++) {
    const uint8_t* y_line = Y_ptr + j * width;
    const uint8_t* u_line = U_ptr + (j / 2) * width_uv;
    const uint8_t* v_line = V_ptr + (j / 2) * width_uv;

    uint8_t* r_out = rgb + 0 * pixels + j * width;
    uint8_t* g_out = rgb + 1 * pixels + j * width;
    uint8_t* b_out = rgb + 2 * pixels + j * width;

    int i = 0;
    for (; i <= width - 8; i += 8) {
      uint8x8_t y_u8 = vld1_u8(y_line + i);
      uint32_t u4, v4;
      memcpy(&u4, u_line + i / 2, 4);
      memcpy(&v4, v_line + i / 2, 4);
      uint8x8_t u_u8_full = vreinterpret_u8_u32(vdup_n_u32(u4));
      uint8x8_t v_u8_full = vreinterpret_u8_u32(vdup_n_u32(v4));

      uint8x8_t u_u8 = vzip_u8(u_u8_full, u_u8_full).val[0];
      uint8x8_t v_u8 = vzip_u8(v_u8_full, v_u8_full).val[0];

      int16x8_t y_s16 = vreinterpretq_s16_u16(vmovl_u8(y_u8));
      int16x8_t C = vsubq_s16(y_s16, v16);
      int16x8_t u_s16 = vreinterpretq_s16_u16(vmovl_u8(u_u8));
      int16x8_t D = vsubq_s16(u_s16, v128);
      int16x8_t v_s16 = vreinterpretq_s16_u16(vmovl_u8(v_u8));
      int16x8_t E = vsubq_s16(v_s16, v128);

      int16x4_t C_lo = vget_low_s16(C);
      int16x4_t C_hi = vget_high_s16(C);
      int16x4_t D_lo = vget_low_s16(D);
      int16x4_t D_hi = vget_high_s16(D);
      int16x4_t E_lo = vget_low_s16(E);
      int16x4_t E_hi = vget_high_s16(E);

      // Low 4 pixels
      int32x4_t c_lo = vmovl_s16(C_lo);
      int32x4_t d_lo = vmovl_s16(D_lo);
      int32x4_t e_lo = vmovl_s16(E_lo);

      int32x4_t r_lo = vmlaq_s32(v128_32, c_lo, c298);
      r_lo = vmlaq_s32(r_lo, e_lo, c409);

      int32x4_t g_lo = vmlaq_s32(v128_32, c_lo, c298);
      g_lo = vmlaq_s32(g_lo, d_lo, c100_neg);
      g_lo = vmlaq_s32(g_lo, e_lo, c208_neg);

      int32x4_t b_lo = vmlaq_s32(v128_32, c_lo, c298);
      b_lo = vmlaq_s32(b_lo, d_lo, c516);

      // High 4 pixels
      int32x4_t c_hi = vmovl_s16(C_hi);
      int32x4_t d_hi = vmovl_s16(D_hi);
      int32x4_t e_hi = vmovl_s16(E_hi);

      int32x4_t r_hi = vmlaq_s32(v128_32, c_hi, c298);
      r_hi = vmlaq_s32(r_hi, e_hi, c409);

      int32x4_t g_hi = vmlaq_s32(v128_32, c_hi, c298);
      g_hi = vmlaq_s32(g_hi, d_hi, c100_neg);
      g_hi = vmlaq_s32(g_hi, e_hi, c208_neg);

      int32x4_t b_hi = vmlaq_s32(v128_32, c_hi, c298);
      b_hi = vmlaq_s32(b_hi, d_hi, c516);

      // Shift & Narrow
      int16x8_t r_16 = vcombine_s16(vshrn_n_s32(r_lo, 8), vshrn_n_s32(r_hi, 8));
      int16x8_t g_16 = vcombine_s16(vshrn_n_s32(g_lo, 8), vshrn_n_s32(g_hi, 8));
      int16x8_t b_16 = vcombine_s16(vshrn_n_s32(b_lo, 8), vshrn_n_s32(b_hi, 8));

      vst1_u8(r_out + i, vqmovun_s16(r_16));
      vst1_u8(g_out + i, vqmovun_s16(g_16));
      vst1_u8(b_out + i, vqmovun_s16(b_16));
    }

    // Scalar cleanup
    for (; i < width; i++) {
      int y_val = Y_ptr[j * width + i];
      int u_val = U_ptr[(j / 2) * width_uv + (i / 2)];
      int v_val = V_ptr[(j / 2) * width_uv + (i / 2)];

      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;

      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;

      r_out[i] = clamp_u8(r);
      g_out[i] = clamp_u8(g);
      b_out[i] = clamp_u8(b);
    }
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*yuv_fn)(const uint8_t*, uint8_t*, int, int);

static double bench(yuv_fn fn, const uint8_t* yuv, uint8_t* rgb, int w, int h) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(yuv, rgb, w, h);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(" YUV420 to RGB v2: Add NEON Vectorized Pipeline (BT.601)\n");
  printf(" Image: %d x %d   Loops: %d\n", width, height, NTIMES);
  printf("===========================================================\n");

  long pixels = (long)width * height;
  // YUV420 planar = Y(pixels) + U(pixels/4) + V(pixels/4) = pixels*3/2
  size_t bytes_yuv = ((size_t)pixels * 3 / 2 + 15) & ~(size_t)15;
  size_t bytes_rgb = ((size_t)pixels * 3 + 15) & ~(size_t)15;

  uint8_t* yuv = (uint8_t*)aligned_alloc(16, bytes_yuv);
  uint8_t* rgb_ref = (uint8_t*)aligned_alloc(16, bytes_rgb);  // float reference
  uint8_t* rgb_test = (uint8_t*)aligned_alloc(16, bytes_rgb);  // working buffer
  if (!yuv || !rgb_ref || !rgb_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialize planar YUV data
  memset(yuv, 0, pixels * 3 / 2);
  for (long i = 0; i < pixels * 3 / 2; i++) yuv[i] = (i % 255);

  // Golden reference = floating-point conversion
  yuv420_to_rgb_float(yuv, rgb_ref, width, height);

  double t_fl = bench(yuv420_to_rgb_float, yuv, rgb_ref, width, height);
  double t_in = bench(yuv420_to_rgb_int, yuv, rgb_test, width, height);
  const char* s_in = check_diff(rgb_ref, rgb_test, pixels);
  double t_ne = bench(yuv420_to_rgb_neon, yuv, rgb_test, width, height);
  const char* s_ne = check_diff(rgb_ref, rgb_test, pixels);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial Float    | %9.3f |    -    |  REF  |\n", t_fl);
  printf("| Serial Int      | %9.3f |  1.00 x |  %-4s |\n", t_in, s_in);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_in / t_ne,
         s_ne);
  printf("-------------------------------------------------\n");

  free(yuv);
  free(rgb_ref);
  free(rgb_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_yuv2rgb/yuv2rgb_v2.c", "src_yuv2rgb/yuv2rgb_v2")
out_v2 = run_bin(BIN, 1920, 1080)

## 7. v3 · 循环展开实现

在 v2 的基础上**新增 `yuv420_to_rgb_neon_unroll` 函数**：单次迭代处理 **16 个像素**，并将低 8、高 8 两组像素并行计算、常量预先算好置于循环外。

### 🔑 知识点
- **循环展开**：单次迭代处理更多像素，减少循环控制开销，隐藏指令延迟
- **提高指令级并行**：低 8 与高 8 两组像素的计算相互独立，可在流水线中重叠执行
- **常量外提**：将各系数向量的构造移出循环，避免重复计算

> 本流程计算密度较高（每像素多次乘加），故向量化与展开的收益较为可观。

至此，**四个实现**（含浮点参考）全部实现，可完整对比浮点、定点、NEON 与展开版本的性能。

In [ ]:
%%writefile src_yuv2rgb/yuv2rgb_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Clamp an integer to the u8 range [0, 255]
static inline uint8_t clamp_u8(int x) {
  if (x < 0) return 0;
  if (x > 255) return 255;
  return (uint8_t)x;
}

// Helper: Verify results against reference (allow +/-2 for float/fixed diff)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n * 3; i++) {
    int diff = abs((int)ref[i] - (int)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff <= 2)
    return "PASS";
  else
    return "FAIL";
}

// 1. Floating-point BT.601 (accurate reference)
void yuv420_to_rgb_float(const uint8_t* yuv, uint8_t* rgb, int width,
                         int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    for (int i = 0; i < width; i++) {
      int y_idx = j * width + i;
      int uv_idx = (j / 2) * (width / 2) + (i / 2);

      float y_val = Y[y_idx];
      float u_val = U[uv_idx];
      float v_val = V[uv_idx];

      // Standard BT.601 conversion
      float c = y_val - 16.0f;
      float d = u_val - 128.0f;
      float e = v_val - 128.0f;

      int r = (int)(1.164f * c + 1.596f * e);
      int g = (int)(1.164f * c - 0.391f * d - 0.813f * e);
      int b = (int)(1.164f * c + 2.018f * d);

      long out_idx = j * width + i;
      // Output: R plane, then G plane, then B plane
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// 2. Serial fixed-point (Q8 integer) - Baseline for speedup
void yuv420_to_rgb_int(const uint8_t* yuv, uint8_t* rgb, int width,
                       int height) {
  long pixels = width * height;
  const uint8_t* Y = yuv;
  const uint8_t* U = yuv + pixels;
  const uint8_t* V = U + (pixels / 4);

  for (int j = 0; j < height; j++) {
    int y_row = j * width;
    int uv_row = (j / 2) * (width / 2);

    for (int i = 0; i < width; i++) {
      int y_val = Y[y_row + i];
      int u_val = U[uv_row + (i / 2)];
      int v_val = V[uv_row + (i / 2)];

      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;

      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;

      long out_idx = y_row + i;
      rgb[0 * pixels + out_idx] = clamp_u8(r);
      rgb[1 * pixels + out_idx] = clamp_u8(g);
      rgb[2 * pixels + out_idx] = clamp_u8(b);
    }
  }
}

// 3. NEON fixed-point: vzip upsample + vmovl widen + vmla + vqmovun saturate
void yuv420_to_rgb_neon(const uint8_t* yuv, uint8_t* rgb, int width,
                        int height) {
  long pixels = width * height;
  const uint8_t* Y_ptr = yuv;
  const uint8_t* U_ptr = yuv + pixels;
  const uint8_t* V_ptr = U_ptr + (pixels / 4);

  // Constants
  const int16x8_t v16 = vdupq_n_s16(16);
  const int16x8_t v128 = vdupq_n_s16(128);
  const int32x4_t v128_32 = vdupq_n_s32(128);

  const int32x4_t c298 = vdupq_n_s32(298);
  const int32x4_t c409 = vdupq_n_s32(409);
  const int32x4_t c100_neg = vdupq_n_s32(-100);
  const int32x4_t c208_neg = vdupq_n_s32(-208);
  const int32x4_t c516 = vdupq_n_s32(516);

  int width_uv = width / 2;

  for (int j = 0; j < height; j++) {
    const uint8_t* y_line = Y_ptr + j * width;
    const uint8_t* u_line = U_ptr + (j / 2) * width_uv;
    const uint8_t* v_line = V_ptr + (j / 2) * width_uv;

    uint8_t* r_out = rgb + 0 * pixels + j * width;
    uint8_t* g_out = rgb + 1 * pixels + j * width;
    uint8_t* b_out = rgb + 2 * pixels + j * width;

    int i = 0;
    for (; i <= width - 8; i += 8) {
      uint8x8_t y_u8 = vld1_u8(y_line + i);
      uint32_t u4, v4;
      memcpy(&u4, u_line + i / 2, 4);
      memcpy(&v4, v_line + i / 2, 4);
      uint8x8_t u_u8_full = vreinterpret_u8_u32(vdup_n_u32(u4));
      uint8x8_t v_u8_full = vreinterpret_u8_u32(vdup_n_u32(v4));

      uint8x8_t u_u8 = vzip_u8(u_u8_full, u_u8_full).val[0];
      uint8x8_t v_u8 = vzip_u8(v_u8_full, v_u8_full).val[0];

      int16x8_t y_s16 = vreinterpretq_s16_u16(vmovl_u8(y_u8));
      int16x8_t C = vsubq_s16(y_s16, v16);
      int16x8_t u_s16 = vreinterpretq_s16_u16(vmovl_u8(u_u8));
      int16x8_t D = vsubq_s16(u_s16, v128);
      int16x8_t v_s16 = vreinterpretq_s16_u16(vmovl_u8(v_u8));
      int16x8_t E = vsubq_s16(v_s16, v128);

      int16x4_t C_lo = vget_low_s16(C);
      int16x4_t C_hi = vget_high_s16(C);
      int16x4_t D_lo = vget_low_s16(D);
      int16x4_t D_hi = vget_high_s16(D);
      int16x4_t E_lo = vget_low_s16(E);
      int16x4_t E_hi = vget_high_s16(E);

      // Low 4 pixels
      int32x4_t c_lo = vmovl_s16(C_lo);
      int32x4_t d_lo = vmovl_s16(D_lo);
      int32x4_t e_lo = vmovl_s16(E_lo);

      int32x4_t r_lo = vmlaq_s32(v128_32, c_lo, c298);
      r_lo = vmlaq_s32(r_lo, e_lo, c409);

      int32x4_t g_lo = vmlaq_s32(v128_32, c_lo, c298);
      g_lo = vmlaq_s32(g_lo, d_lo, c100_neg);
      g_lo = vmlaq_s32(g_lo, e_lo, c208_neg);

      int32x4_t b_lo = vmlaq_s32(v128_32, c_lo, c298);
      b_lo = vmlaq_s32(b_lo, d_lo, c516);

      // High 4 pixels
      int32x4_t c_hi = vmovl_s16(C_hi);
      int32x4_t d_hi = vmovl_s16(D_hi);
      int32x4_t e_hi = vmovl_s16(E_hi);

      int32x4_t r_hi = vmlaq_s32(v128_32, c_hi, c298);
      r_hi = vmlaq_s32(r_hi, e_hi, c409);

      int32x4_t g_hi = vmlaq_s32(v128_32, c_hi, c298);
      g_hi = vmlaq_s32(g_hi, d_hi, c100_neg);
      g_hi = vmlaq_s32(g_hi, e_hi, c208_neg);

      int32x4_t b_hi = vmlaq_s32(v128_32, c_hi, c298);
      b_hi = vmlaq_s32(b_hi, d_hi, c516);

      // Shift & Narrow
      int16x8_t r_16 = vcombine_s16(vshrn_n_s32(r_lo, 8), vshrn_n_s32(r_hi, 8));
      int16x8_t g_16 = vcombine_s16(vshrn_n_s32(g_lo, 8), vshrn_n_s32(g_hi, 8));
      int16x8_t b_16 = vcombine_s16(vshrn_n_s32(b_lo, 8), vshrn_n_s32(b_hi, 8));

      vst1_u8(r_out + i, vqmovun_s16(r_16));
      vst1_u8(g_out + i, vqmovun_s16(g_16));
      vst1_u8(b_out + i, vqmovun_s16(b_16));
    }

    // Scalar cleanup
    for (; i < width; i++) {
      int y_val = Y_ptr[j * width + i];
      int u_val = U_ptr[(j / 2) * width_uv + (i / 2)];
      int v_val = V_ptr[(j / 2) * width_uv + (i / 2)];

      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;

      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;

      r_out[i] = clamp_u8(r);
      g_out[i] = clamp_u8(g);
      b_out[i] = clamp_u8(b);
    }
  }
}

// 4. NEON unrolled: process 16 pixels per iteration
void yuv420_to_rgb_neon_unroll(const uint8_t* yuv, uint8_t* rgb, int width,
                               int height) {
  long pixels = width * height;
  const uint8_t* Y_ptr = yuv;
  const uint8_t* U_ptr = yuv + pixels;
  const uint8_t* V_ptr = U_ptr + (pixels / 4);

  // Constants (Same as before)
  const int16x8_t v16 = vdupq_n_s16(16);
  const int16x8_t v128 = vdupq_n_s16(128);
  const int32x4_t v128_32 = vdupq_n_s32(128);

  const int32x4_t c298 = vdupq_n_s32(298);
  const int32x4_t c409 = vdupq_n_s32(409);
  const int32x4_t c100_neg = vdupq_n_s32(-100);
  const int32x4_t c208_neg = vdupq_n_s32(-208);
  const int32x4_t c516 = vdupq_n_s32(516);

  int width_uv = width / 2;

  for (int j = 0; j < height; j++) {
    const uint8_t* y_line = Y_ptr + j * width;
    const uint8_t* u_line = U_ptr + (j / 2) * width_uv;
    const uint8_t* v_line = V_ptr + (j / 2) * width_uv;

    uint8_t* r_out = rgb + 0 * pixels + j * width;
    uint8_t* g_out = rgb + 1 * pixels + j * width;
    uint8_t* b_out = rgb + 2 * pixels + j * width;

    int i = 0;
    // [Optimization] Unroll: Process 16 pixels at a time
    for (; i <= width - 16; i += 16) {
      // --- Prefetch Next Chunk ---
      __builtin_prefetch(y_line + i + 16);
      __builtin_prefetch(u_line + i / 2 + 8);

      // ==========================
      // BLOCK A: First 8 Pixels
      // ==========================

      // 1. Load Data (Block A)
      uint8x8_t y_u8_a = vld1_u8(y_line + i);
      // U/V: Load 8 bytes (enough for 16 Y pixels), split later
      uint8x8_t u_u8_all = vld1_u8(u_line + i / 2);
      uint8x8_t v_u8_all = vld1_u8(v_line + i / 2);

      // Use vzip to expand U/V for both blocks:
      // u_u8_all contains [u0, u1, u2, u3, u4, u5, u6, u7]
      // vzip.val[0] -> [u0, u0, u1, u1, u2, u2, u3, u3] (Perfect for Block A)
      // vzip.val[1] -> [u4, u4, u5, u5, u6, u6, u7, u7] (Perfect for Block B)
      uint8x8x2_t u_zip = vzip_u8(u_u8_all, u_u8_all);
      uint8x8x2_t v_zip = vzip_u8(v_u8_all, v_u8_all);

      // Block A Operations
      int16x8_t y_s16_a = vreinterpretq_s16_u16(vmovl_u8(y_u8_a));
      int16x8_t C_a = vsubq_s16(y_s16_a, v16);
      int16x8_t u_s16_a = vreinterpretq_s16_u16(vmovl_u8(u_zip.val[0]));
      int16x8_t D_a = vsubq_s16(u_s16_a, v128);
      int16x8_t v_s16_a = vreinterpretq_s16_u16(vmovl_u8(v_zip.val[0]));
      int16x8_t E_a = vsubq_s16(v_s16_a, v128);

      // ==========================
      // BLOCK B: Next 8 Pixels (Interleaved execution)
      // ==========================
      uint8x8_t y_u8_b = vld1_u8(y_line + i + 8);

      int16x8_t y_s16_b = vreinterpretq_s16_u16(vmovl_u8(y_u8_b));
      int16x8_t C_b = vsubq_s16(y_s16_b, v16);
      int16x8_t u_s16_b =
          vreinterpretq_s16_u16(vmovl_u8(u_zip.val[1]));  // Use high part
      int16x8_t D_b = vsubq_s16(u_s16_b, v128);
      int16x8_t v_s16_b =
          vreinterpretq_s16_u16(vmovl_u8(v_zip.val[1]));  // Use high part
      int16x8_t E_b = vsubq_s16(v_s16_b, v128);

      // --- Calculation Block A (Macros/Inline for brevity could help
      // readability) --- Low 4
      int32x4_t c_lo_a = vmovl_s16(vget_low_s16(C_a));
      int32x4_t d_lo_a = vmovl_s16(vget_low_s16(D_a));
      int32x4_t e_lo_a = vmovl_s16(vget_low_s16(E_a));

      int32x4_t r_lo_a = vmlaq_s32(v128_32, c_lo_a, c298);
      r_lo_a = vmlaq_s32(r_lo_a, e_lo_a, c409);
      int32x4_t g_lo_a = vmlaq_s32(v128_32, c_lo_a, c298);
      g_lo_a = vmlaq_s32(g_lo_a, d_lo_a, c100_neg);
      g_lo_a = vmlaq_s32(g_lo_a, e_lo_a, c208_neg);
      int32x4_t b_lo_a = vmlaq_s32(v128_32, c_lo_a, c298);
      b_lo_a = vmlaq_s32(b_lo_a, d_lo_a, c516);

      // High 4
      int32x4_t c_hi_a = vmovl_s16(vget_high_s16(C_a));
      int32x4_t d_hi_a = vmovl_s16(vget_high_s16(D_a));
      int32x4_t e_hi_a = vmovl_s16(vget_high_s16(E_a));

      int32x4_t r_hi_a = vmlaq_s32(v128_32, c_hi_a, c298);
      r_hi_a = vmlaq_s32(r_hi_a, e_hi_a, c409);
      int32x4_t g_hi_a = vmlaq_s32(v128_32, c_hi_a, c298);
      g_hi_a = vmlaq_s32(g_hi_a, d_hi_a, c100_neg);
      g_hi_a = vmlaq_s32(g_hi_a, e_hi_a, c208_neg);
      int32x4_t b_hi_a = vmlaq_s32(v128_32, c_hi_a, c298);
      b_hi_a = vmlaq_s32(b_hi_a, d_hi_a, c516);

      // --- Calculation Block B (Interleaved) ---
      // Low 4
      int32x4_t c_lo_b = vmovl_s16(vget_low_s16(C_b));
      int32x4_t d_lo_b = vmovl_s16(vget_low_s16(D_b));
      int32x4_t e_lo_b = vmovl_s16(vget_low_s16(E_b));

      int32x4_t r_lo_b = vmlaq_s32(v128_32, c_lo_b, c298);
      r_lo_b = vmlaq_s32(r_lo_b, e_lo_b, c409);
      int32x4_t g_lo_b = vmlaq_s32(v128_32, c_lo_b, c298);
      g_lo_b = vmlaq_s32(g_lo_b, d_lo_b, c100_neg);
      g_lo_b = vmlaq_s32(g_lo_b, e_lo_b, c208_neg);
      int32x4_t b_lo_b = vmlaq_s32(v128_32, c_lo_b, c298);
      b_lo_b = vmlaq_s32(b_lo_b, d_lo_b, c516);

      // High 4
      int32x4_t c_hi_b = vmovl_s16(vget_high_s16(C_b));
      int32x4_t d_hi_b = vmovl_s16(vget_high_s16(D_b));
      int32x4_t e_hi_b = vmovl_s16(vget_high_s16(E_b));

      int32x4_t r_hi_b = vmlaq_s32(v128_32, c_hi_b, c298);
      r_hi_b = vmlaq_s32(r_hi_b, e_hi_b, c409);
      int32x4_t g_hi_b = vmlaq_s32(v128_32, c_hi_b, c298);
      g_hi_b = vmlaq_s32(g_hi_b, d_hi_b, c100_neg);
      g_hi_b = vmlaq_s32(g_hi_b, e_hi_b, c208_neg);
      int32x4_t b_hi_b = vmlaq_s32(v128_32, c_hi_b, c298);
      b_hi_b = vmlaq_s32(b_hi_b, d_hi_b, c516);

      // --- Store Block A ---
      int16x8_t r_16_a =
          vcombine_s16(vshrn_n_s32(r_lo_a, 8), vshrn_n_s32(r_hi_a, 8));
      int16x8_t g_16_a =
          vcombine_s16(vshrn_n_s32(g_lo_a, 8), vshrn_n_s32(g_hi_a, 8));
      int16x8_t b_16_a =
          vcombine_s16(vshrn_n_s32(b_lo_a, 8), vshrn_n_s32(b_hi_a, 8));

      vst1_u8(r_out + i, vqmovun_s16(r_16_a));
      vst1_u8(g_out + i, vqmovun_s16(g_16_a));
      vst1_u8(b_out + i, vqmovun_s16(b_16_a));

      // --- Store Block B ---
      int16x8_t r_16_b =
          vcombine_s16(vshrn_n_s32(r_lo_b, 8), vshrn_n_s32(r_hi_b, 8));
      int16x8_t g_16_b =
          vcombine_s16(vshrn_n_s32(g_lo_b, 8), vshrn_n_s32(g_hi_b, 8));
      int16x8_t b_16_b =
          vcombine_s16(vshrn_n_s32(b_lo_b, 8), vshrn_n_s32(b_hi_b, 8));

      vst1_u8(r_out + i + 8, vqmovun_s16(r_16_b));
      vst1_u8(g_out + i + 8, vqmovun_s16(g_16_b));
      vst1_u8(b_out + i + 8, vqmovun_s16(b_16_b));
    }

    // Scalar cleanup
    for (; i < width; i++) {
      // ... (Duplicate scalar code here or call helper) ...
      int y_val = Y_ptr[j * width + i];
      int u_val = U_ptr[(j / 2) * width_uv + (i / 2)];
      int v_val = V_ptr[(j / 2) * width_uv + (i / 2)];
      int C = y_val - 16;
      int D = u_val - 128;
      int E = v_val - 128;
      int r = (298 * C + 409 * E + 128) >> 8;
      int g = (298 * C - 100 * D - 208 * E + 128) >> 8;
      int b = (298 * C + 516 * D + 128) >> 8;
      r_out[i] = clamp_u8(r);
      g_out[i] = clamp_u8(g);
      b_out[i] = clamp_u8(b);
    }
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*yuv_fn)(const uint8_t*, uint8_t*, int, int);

static double bench(yuv_fn fn, const uint8_t* yuv, uint8_t* rgb, int w, int h) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(yuv, rgb, w, h);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(" YUV420 to RGB v3: Add NEON Unrolled (BT.601)\n");
  printf(" Image: %d x %d   Loops: %d\n", width, height, NTIMES);
  printf("===========================================================\n");

  long pixels = (long)width * height;
  // YUV420 planar = Y(pixels) + U(pixels/4) + V(pixels/4) = pixels*3/2
  size_t bytes_yuv = ((size_t)pixels * 3 / 2 + 15) & ~(size_t)15;
  size_t bytes_rgb = ((size_t)pixels * 3 + 15) & ~(size_t)15;

  uint8_t* yuv = (uint8_t*)aligned_alloc(16, bytes_yuv);
  uint8_t* rgb_ref = (uint8_t*)aligned_alloc(16, bytes_rgb);  // float reference
  uint8_t* rgb_test = (uint8_t*)aligned_alloc(16, bytes_rgb);  // working buffer
  if (!yuv || !rgb_ref || !rgb_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialize planar YUV data
  memset(yuv, 0, pixels * 3 / 2);
  for (long i = 0; i < pixels * 3 / 2; i++) yuv[i] = (i % 255);

  // Golden reference = floating-point conversion
  yuv420_to_rgb_float(yuv, rgb_ref, width, height);

  double t_fl = bench(yuv420_to_rgb_float, yuv, rgb_ref, width, height);
  double t_in = bench(yuv420_to_rgb_int, yuv, rgb_test, width, height);
  const char* s_in = check_diff(rgb_ref, rgb_test, pixels);
  double t_ne = bench(yuv420_to_rgb_neon, yuv, rgb_test, width, height);
  const char* s_ne = check_diff(rgb_ref, rgb_test, pixels);
  double t_un = bench(yuv420_to_rgb_neon_unroll, yuv, rgb_test, width, height);
  const char* s_un = check_diff(rgb_ref, rgb_test, pixels);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial Float    | %9.3f |    -    |  REF  |\n", t_fl);
  printf("| Serial Int      | %9.3f |  1.00 x |  %-4s |\n", t_in, s_in);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_in / t_ne,
         s_ne);
  printf("| NEON Unrolled   | %9.3f | %5.2f x |  %-4s |\n", t_un, t_in / t_un,
         s_un);
  printf("-------------------------------------------------\n");

  free(yuv);
  free(rgb_ref);
  free(rgb_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_yuv2rgb/yuv2rgb_v3.c", "src_yuv2rgb/yuv2rgb_v3")
out_v3 = run_bin(BIN, 1920, 1080)

## 8. 📈 性能可视化（基于 v3 的四版本结果）

v3 的输出包含全部四个实现的耗时与加速比，据此绘制柱状图，以完整呈现各手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v3 = parse_table(out_v3)
for r in rows_v3:
    sp = f'{r["speedup"]:.2f}x' if r["speedup"] is not None else "  -  "
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {sp}')
# Serial Float 为参考(REF)、无 speedup 数值，绘图时会被自动跳过
plot_speedup([r for r in rows_v3 if r["speedup"] is not None], "YUV420 to RGB: performance of implementations (1920x1080)")

## 9. 结果分析

> 注：具体数值随硬件平台、图像尺寸、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

**① Serial Float 与 Serial Int 耗时接近。**

现代处理器配有硬件 FPU，标量层面的浮点乘加与整数乘加吞吐相近，因此定点化在此并未带来明显提速。这表明：**定点化本身并不是标量层面的优化手段。**

**② 一旦采用 NEON，加速随即显现。**

其原因在于：**SIMD 对 8/16 位整数的并行度远高于浮点**——一个 128 位向量可容纳 8~16 个整数，而仅能容纳 4 个 32 位浮点；且整数运算省电、不依赖 FPU。因此，定点化的真正价值在于**使高并行度的向量化成为可能**。

**③ 循环展开带来进一步的稳定提升。**

本流程计算密度较高，展开通过隐藏指令延迟、提高指令级并行，可在 NEON 基础上再获得一定收益。

---

### 🎓 结论
YUV→RGB 将本章**全部技术**（色度上采样 `vzip`、类型提升 `vmovl`、定点乘加、饱和窄化 `vqmovun`、无分支处理）整合于一条真实的视频解码流程之中，并揭示了一条重要的工程判断：

> **定点化的意义不在标量层面的提速，而在于为向量化创造条件。** 优化的一般次序为：**先定点化、再向量化、后循环展开**。

🎓 **至此，本章六个实验全部完成。** 从 AXPY（L1，访存受限）、GEMV（L2，规约）、图像处理系列（结构化访存、定点、饱和、无分支），到本综合案例，学生已建立起“**分析瓶颈 → 选择优化范式 → 以实测验证**”的完整方法。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察结果的变化（建议先独立完成，再阅读思考题）：

1. 将 BT.601 的定点系数替换为 **BT.709**（高清标准，如 R 的系数为 1.793 等），观察输出颜色的变化，理解“转换标准必须与数据源一致”。
2. 在 `main` 中再增加一个纯浮点计时项与定点计时项的对比，进一步验证“标量层面定点化不省时间”。
3. 将上采样由 `vzip` 改为标量的最近邻复制写法，对比速度，体会结构化指令的价值。
4. 【拓展】该算法还有 **Shift Approx**（移位近似）与 **LUT 查表** 两种实现，可将其加入对比，分析它们各自的适用场景与代价。

## 11. 🤔 思考题

- 定点化在标量层面几乎不省时间，为何一旦采用 NEON 就能获得大幅提速？
- 为何 G 通道的公式最复杂（含两个减项）？这与 YUV 的定义有何关系？
- `vqmovun` 的“无分支饱和”相比浮点版本中的 `if` 钳位，为何在性能上更有利？
- 色度为何要先上采样才能与亮度对齐？若不上采样直接计算会有什么问题？
- 综合本章六个实验，“是否需要手写 NEON”的判断依据是什么？

## 12. 小结与展望

本实验完成了 YUV420→RGB 从串行到 NEON 的实现过程：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>yuv420_to_rgb_float</code> + <code>yuv420_to_rgb_int</code></td>
      <td style="text-align: left;">浮点参考、定点化、精度与速度的关系</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>yuv420_to_rgb_neon</code></td>
      <td style="text-align: left;">上采样、类型提升、定点乘加、饱和窄化、无分支</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>yuv420_to_rgb_neon_unroll</code></td>
      <td style="text-align: left;">循环展开、指令级并行</td>
    </tr>
  </tbody>
</table>

YUV→RGB 作为本章的综合案例，整合了全部核心技术，并阐明了“**定点化为向量化服务**”这一关键工程直觉。

### 🎓 全章回顾
从 **AXPY（L1，访存受限）→ GEMV（L2，规约与数据复用）→ 图像处理系列（结构化访存、定点、饱和、无分支）→ YUV→RGB（综合流程）**，本章系统训练了一套优化方法。

➡️ **进一步的方向**：本章止于 L1/L2 级别的运算；真正能逼近处理器算力峰值的是 **L3 的通用矩阵乘（GEMM）**——那是缓存分块与数据复用的主要场景，也是后续深入学习的方向。